# Data Download: SDSS Spectra + WISE Photometry
### Jespersen et al. (2025) — *"The optical and infrared are connected"*

---

## Purpose

This notebook downloads everything needed to run the preprocessing and MLP pipelines,
**without relying on CasJobs or rsync** (both of which have access issues).

It uses two methods:

| What | How | Output |
|---|---|---|
| SDSS galaxy metadata (plate, MJD, fiber, z, …) | Direct HTTP to SDSS SkyServer SQL API | `sdss_raw/sdss_metadata.csv` |
| SDSS FITS spectra (one file per galaxy) | Parallel HTTP from SDSS SAS | `sdss_raw/fits/{plate:04d}/spec-*.fits` |
| WISE cross-match photometry | Direct HTTP to SDSS SkyServer SQL API | `wise_raw/wise_crossmatch.csv` |

---

## Why not CasJobs / rsync?

- **CasJobs** (skyserver.sdss.org/casjobs) migrated to SciServer and no longer has a
  simple paste-SQL interface.
- **rsync://data.sdss.org** is blocked by firewalls at many institutions.
- The **SDSS SkyServer HTTP SQL API** (`skyserver.sdss.org/drNN/SkyServerWS/SearchTools/SqlSearch`)
  is the same backend that CasJobs used — it still works fine and returns CSV directly.
  We just need to paginate ourselves for > ~100k rows.
- The **SDSS Science Archive Server** (`dr17.sdss.org/sas/…`) serves individual FITS files
  over HTTPS, which works everywhere HTTP works.

---

## Run order

```
data_download.ipynb          ← run this first
    ↓ sdss_raw/sdss_metadata.csv
    ↓ sdss_raw/fits/**/*.fits
    ↓ wise_raw/wise_crossmatch.csv
sdss_data_preprocessing.ipynb   ← set DATA_SOURCE = "local_fits"
wise_data_preprocessing.ipynb   ← set DATA_SOURCE = "csv"
reproduce_figure3_ppplot.ipynb
```


## Cell 0 — Imports

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import requests
import io, time, sys, logging, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("Download")

print(f"Python {sys.version.split()[0]}  |  requests {requests.__version__}")
print("All imports OK ✓")


Python 3.11.14  |  requests 2.32.5
All imports OK ✓


## Cell 1 — Configuration

In [2]:
# ═══════════════════════════════════════════════════════════════════════
# SCALE KNOBS
# ═══════════════════════════════════════════════════════════════════════
N_SPECTRA        = 500     # None → full ~510k dataset.
                            # Start with 500–1000 to verify everything works,
                            # then set to None for the full run.

N_WORKERS        = 32      # Parallel download threads for FITS files.
                            # On your 768-core server you could use 128+,
                            # but the SDSS server may throttle above ~64.
                            # Start with 32; increase if it's fast enough.

FITS_DR          = 17      # SDSS data release for FITS files.
                            # DR17 = final SDSS-II legacy release (paper uses this).
                            # DR19 uses a different directory structure (see Cell 5).

SQL_DR           = 18      # Data release for SQL queries.
                            # DR18 has the same MGS + WISE tables as DR17.
                            # Kept separate because DR18 SQL server is more stable.

PAGE_SIZE        = 100_000 # Rows per SQL page. The SkyServer HTTP API can handle
                            # up to 500k but smaller pages are safer and resume better.
# ═══════════════════════════════════════════════════════════════════════

SDSS_SQL_URL = f"https://skyserver.sdss.org/dr{SQL_DR}/SkyServerWS/SearchTools/SqlSearch"
SDSS_FITS_BASE = f"https://dr{FITS_DR}.sdss.org/sas/dr{FITS_DR}/sdss/spectro/redux/26/spectra/lite"

Z_MAX = 0.5

# Output directories
SDSS_RAW_DIR = Path("sdss_raw")
WISE_RAW_DIR = Path("wise_raw")
FITS_DIR     = SDSS_RAW_DIR / "fits"

SDSS_RAW_DIR.mkdir(exist_ok=True)
WISE_RAW_DIR.mkdir(exist_ok=True)
FITS_DIR.mkdir(exist_ok=True)

SDSS_META_CSV = SDSS_RAW_DIR / "sdss_metadata.csv"
WISE_CSV      = WISE_RAW_DIR / "wise_crossmatch.csv"

log.info(f"Config: N_SPECTRA={N_SPECTRA}, N_WORKERS={N_WORKERS}, DR{FITS_DR}")
log.info(f"SDSS SQL: {SDSS_SQL_URL}")
log.info(f"SDSS FITS: {SDSS_FITS_BASE}/{{plate:04d}}/spec-*.fits")


14:20:21 [INFO] Config: N_SPECTRA=500, N_WORKERS=32, DR17
14:20:21 [INFO] SDSS SQL: https://skyserver.sdss.org/dr18/SkyServerWS/SearchTools/SqlSearch
14:20:21 [INFO] SDSS FITS: https://dr17.sdss.org/sas/dr17/sdss/spectro/redux/26/spectra/lite/{plate:04d}/spec-*.fits


## Cell 2 — SQL Helpers (Paginated HTTP)

### Why pagination?

The SkyServer HTTP API has an implicit row limit (it varies by DR, but ~500k is common).
For the full ~510k galaxy sample, we might hit this. More importantly, paginated queries
are **resumable**: if the download is interrupted, we can restart from the last completed
page rather than re-running the whole query.

### Pagination strategy

Instead of SQL `OFFSET` (which forces a full table scan for each page — O(N²) cost),
we use a **cursor-based** approach: track the maximum `specobjid` seen so far and add
`WHERE specobjid > {cursor}` to each subsequent page. This works in O(1) per page
because `specobjid` is a primary key with an index.

Analogy from computer science: this is exactly like B-tree cursor pagination —
the same technique used by database ORMs for efficient large-result pagination.


In [ ]:
def sql_query_http(sql, dr=SQL_DR, timeout=300):
    """
    Send a single SQL query to the SDSS SkyServer HTTP API.

    Parameters
    ----------
    sql : str
        Complete SQL query string.
    dr : int
        SDSS data release number.
    timeout : int
        HTTP timeout in seconds.

    Returns
    -------
    pd.DataFrame or None
        DataFrame of results, or None on failure.
    """
    url = f"https://skyserver.sdss.org/dr{dr}/SkyServerWS/SearchTools/SqlSearch"
    try:
        resp = requests.get(
            url,
            params={"cmd": sql, "format": "csv"},
            timeout=timeout,
        )
        text = resp.text.strip()

        # The API returns HTML error pages on failure, not HTTP error codes.
        # Detect these by checking for HTML tags or the word "error".
        if resp.status_code != 200:
            log.warning(f"HTTP {resp.status_code}: {text[:200]}")
            return None
        # Check for HTML error pages (the API returns these instead of HTTP error codes)
        # We look for "<html" and common SDSS error strings, but NOT generic "error"
        # because column names like "w1sigmpro" are fine.
        if ("<html" in text[:200].lower()
                or "syntax error" in text[:300].lower()
                or "server error" in text[:300].lower()
                or text.strip() == ""):
            log.warning(f"Server returned error: {text[:300]}")
            return None
        if not text or text == "":
            return None

        df = pd.read_csv(io.StringIO(text), comment="#")  # SDSS API prepends "#Table1" header line
        return df if len(df) > 0 else None

    except requests.exceptions.Timeout:
        log.warning(f"Query timed out after {timeout}s")
        return None
    except Exception as e:
        log.warning(f"Query failed: {type(e).__name__}: {e}")
        return None


def sql_query_paginated(sql_template, id_col="specobjid", cursor_col=None,
                        page_size=PAGE_SIZE, max_rows=None, dr=SQL_DR, verbose=True):
    """
    Execute a potentially large SQL query in cursor-paginated chunks.

    Parameters
    ----------
    sql_template : str
        SQL string with two placeholders:
          {page_size}  — replaced with the TOP N limit
          {cursor}     — replaced with WHERE {cursor_col} > {cursor} clause
        Example:
          SELECT TOP {page_size} s.specobjid, ...
          FROM SpecObj AS s JOIN PhotoObj AS p ON ...
          WHERE {cursor} AND s.z < 0.5
          ORDER BY s.specobjid
    id_col : str
        Column name in the returned DataFrame used to advance the cursor.
        This is the unqualified name as it appears in query results (e.g. "specobjid").
    cursor_col : str or None
        Qualified column name to use in the WHERE clause (e.g. "s.specobjid").
        Defaults to id_col if not provided. Use this when the query JOINs
        multiple tables and the unqualified name would be ambiguous.
    page_size : int
        Rows per page.
    max_rows : int or None
        Stop after collecting this many rows total.
    dr : int
        SDSS data release.
    verbose : bool
        Log progress.

    Returns
    -------
    pd.DataFrame
        All results concatenated.
    """
    if cursor_col is None:
        cursor_col = id_col

    pages = []
    cursor = 0         # The cursor starts at 0 (i.e., WHERE specobjid > 0 = all rows)
    total  = 0
    page_n = 0

    while True:
        # Replace placeholders in the template
        cursor_clause = f"{cursor_col} > {cursor}"
        sql = sql_template.format(page_size=page_size, cursor=cursor_clause)

        t0 = time.time()
        df_page = sql_query_http(sql, dr=dr)
        elapsed = time.time() - t0

        if df_page is None or len(df_page) == 0:
            break   # No more rows — we've fetched everything

        pages.append(df_page)
        total  += len(df_page)
        page_n += 1

        # Advance cursor to the maximum id in this page
        cursor = int(df_page[id_col].max())

        if verbose:
            log.info(f"  Page {page_n}: {len(df_page):,} rows in {elapsed:.1f}s  "
                     f"(total={total:,}, cursor={cursor})")

        # If this page was smaller than page_size, we've hit the end
        if len(df_page) < page_size:
            break

        if max_rows is not None and total >= max_rows:
            break

    if not pages:
        return pd.DataFrame()

    result = pd.concat(pages, ignore_index=True)
    result.columns = result.columns.str.lower().str.strip()

    if max_rows is not None:
        result = result.head(max_rows)

    if verbose:
        log.info(f"Total rows fetched: {len(result):,}")
    return result


# ── Quick connectivity test ─────────────────────────────────────────────
log.info("Testing SkyServer SQL API connectivity ...")
test_df = sql_query_http(
    "SELECT TOP 3 specobjid, z FROM SpecObj WHERE class='GALAXY' ORDER BY specobjid",
    dr=SQL_DR,
)
if test_df is not None and len(test_df) > 0:
    print(f"✓ SQL API (DR{SQL_DR}) is reachable — got {len(test_df)} test rows")
    print(test_df.to_string(index=False))
else:
    print(f"✗ SQL API (DR{SQL_DR}) is NOT reachable.")
    print("  This notebook requires internet access to skyserver.sdss.org.")
    print("  If you are behind a proxy, set the https_proxy environment variable.")

## Cell 3 — Download SDSS Galaxy Metadata (SQL → CSV)

This downloads the selection table used to build the galaxy sample.
We do NOT download the spectra here — just the metadata needed to
identify which FITS files to fetch.

The query applies the same cuts as the paper (§2.1):
- `class = 'GALAXY'`, z ∈ [0.01, 0.5], `zWarning = 0`
- `petroMag_r < 17.8` (Main Galaxy Sample bright limit)

This is the same SQL that was in CasJobs — we're just running it
directly against the HTTP API instead.


In [ ]:
def build_sdss_sql_paged(z_max=0.5):
    """
    Return an SQL template for paginated SDSS galaxy metadata retrieval.

    The template contains two placeholders consumed by sql_query_paginated():
      {page_size}  — the TOP N limit
      {cursor}     — the cursor clause (s.specobjid > X)

    Note: `ORDER BY s.specobjid` is mandatory for cursor pagination.
    """
    return f"""SELECT TOP {{page_size}}
    s.specobjid, s.plate, s.mjd, s.fiberid,
    s.z AS redshift, s.zwarning,
    s.class AS specclass, s.subclass AS specsubclass,
    p.petroMag_r
FROM SpecObj AS s
JOIN PhotoObj AS p ON s.bestobjid = p.objid
WHERE {{cursor}}
  AND s.class = 'GALAXY'
  AND s.z BETWEEN 0.01 AND {z_max}
  AND s.zwarning = 0
  AND p.petroMag_r BETWEEN 14.0 AND 17.8
  AND p.petroMag_r != -9999
ORDER BY s.specobjid"""


# ── Check if metadata already downloaded (for resume support) ─────────
if SDSS_META_CSV.exists():
    existing = pd.read_csv(SDSS_META_CSV)
    log.info(f"Found existing {SDSS_META_CSV}: {len(existing):,} rows — reusing.")
    log.info(f"  Delete {SDSS_META_CSV} to re-download.")
    df_sdss = existing
else:
    log.info(f"Downloading SDSS metadata (N_SPECTRA={N_SPECTRA}) ...")
    t0 = time.time()

    sql_tmpl = build_sdss_sql_paged(z_max=Z_MAX)
    df_sdss = sql_query_paginated(
        sql_tmpl,
        id_col="specobjid",
        cursor_col="s.specobjid",   # qualified name avoids ambiguity in the JOIN
        page_size=PAGE_SIZE,
        max_rows=N_SPECTRA,   # None = all
        dr=SQL_DR,
    )

    if len(df_sdss) == 0:
        raise RuntimeError(
            "SDSS metadata query returned no rows.\n"
            "Check: is skyserver.sdss.org reachable from this machine?\n"
            "Try:   curl 'https://skyserver.sdss.org/dr18/SkyServerWS/"
            "SearchTools/SqlSearch?cmd=SELECT+TOP+2+specobjid+FROM+SpecObj"
            "+WHERE+class%3D%27GALAXY%27&format=csv'"
        )

    df_sdss.to_csv(SDSS_META_CSV, index=False)
    log.info(f"Saved {len(df_sdss):,} rows to {SDSS_META_CSV} in {time.time()-t0:.1f}s")

print(f"\nSDSS metadata: {len(df_sdss):,} galaxies × {len(df_sdss.columns)} columns")
print(df_sdss.head(3).to_string())

## Cell 4 — Metadata Diagnostic

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

print(f"Redshift range: [{df_sdss['redshift'].min():.4f}, {df_sdss['redshift'].max():.4f}]")
print(f"petroMag_r range: [{df_sdss['petromag_r'].min():.2f}, {df_sdss['petromag_r'].max():.2f}]")
n_unique_plates = df_sdss["plate"].nunique()
print(f"Unique plates: {n_unique_plates:,}  (each plate = 640 fibres)")
print(f"Estimated total FITS files: {len(df_sdss):,}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_sdss["redshift"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Redshift z"); axes[0].set_ylabel("Count")
axes[0].set_title("Redshift distribution")

axes[1].hist(df_sdss["petromag_r"], bins=50, color="mediumpurple", edgecolor="white")
axes[1].set_xlabel("petroMag_r (AB)"); axes[1].set_ylabel("Count")
axes[1].set_title("r-band magnitude distribution")

plate_counts = df_sdss["plate"].value_counts()
axes[2].hist(plate_counts.values, bins=40, color="darkorange", edgecolor="white")
axes[2].set_xlabel("Galaxies per plate"); axes[2].set_ylabel("# plates")
axes[2].set_title(f"Plate occupancy ({n_unique_plates:,} plates total)")

plt.tight_layout()
plt.savefig(SDSS_RAW_DIR / "metadata_diagnostics.png", dpi=130, bbox_inches="tight")
plt.show()
log.info(f"Saved diagnostics to {SDSS_RAW_DIR}/metadata_diagnostics.png")


## Cell 5 — Download SDSS FITS Spectra (Parallel)

### File naming convention

SDSS organises its spectra as:
```
{base}/
  {plate:04d}/
    spec-{plate:04d}-{mjd:05d}-{fiber:04d}.fits
```

Each `spec-*.fits` file (the "lite" version) is ~150–300 KB and contains:
- **Extension 0**: primary header with observation metadata
- **Extension 1**: FITS binary table with columns `loglam`, `flux`, `ivar`, `and_mask`, etc.
- **Extension 2**: Per-frame metadata (not needed for preprocessing)

### Download strategy

We download in parallel using Python's `ThreadPoolExecutor`. Each worker
downloads one file, saves it to the correct subdirectory, and returns a
status code. The main thread tracks progress and retries failures.

**Resume support**: before attempting a download, we check whether the target
file already exists with a non-zero size. This means you can kill the job at
any time and restart — already-downloaded files are skipped.

### DR17 vs DR19

The user's server can reach `https://dr19.sdss.org/sas/dr19/…`. DR19 uses
a different directory structure than DR17:
  - DR17: `.../redux/26/spectra/lite/{plate:04d}/spec-*.fits`
  - DR19: `.../spectro/sdss/redux/{run2d}/spectra/lite/{plate:04d}/spec-*.fits`
  where `run2d` is typically `v5_13_2` or similar.

The code below uses DR17 (set `FITS_DR = 17` in Cell 1), which is cleaner and
matches the paper's data. If you find DR17 unreachable from your machine but
DR19 is reachable, change `FITS_DR = 19` and the URL builder below accordingly.


In [ ]:
def fits_url(plate, mjd, fiber, dr=FITS_DR):
    """
    Build the URL for a single SDSS spec-lite FITS file.

    DR17 URL structure:
        https://dr17.sdss.org/sas/dr17/sdss/spectro/redux/26/spectra/lite/
        {plate:04d}/spec-{plate:04d}-{mjd:05d}-{fiber:04d}.fits

    DR19 uses a slightly different path — if DR17 is unreachable, change
    FITS_DR = 19 in Cell 1 and update this function as follows:
        base = f'https://dr19.sdss.org/sas/dr19/spectro/sdss/redux/v5_13_2/spectra/lite'
        (the run2d tag v5_13_2 may vary; check dr19.sdss.org/sas/dr19/spectro/sdss/redux/)
    """
    p4  = f"{plate:04d}"
    m5  = f"{mjd:05d}"
    f4  = f"{fiber:04d}"
    fname = f"spec-{p4}-{m5}-{f4}.fits"
    return f"https://dr{dr}.sdss.org/sas/dr{dr}/sdss/spectro/redux/26/spectra/lite/{p4}/{fname}"


def fits_local_path(plate, mjd, fiber):
    """Local destination for a FITS file."""
    p4 = f"{plate:04d}"
    m5 = f"{mjd:05d}"
    f4 = f"{fiber:04d}"
    return FITS_DIR / p4 / f"spec-{p4}-{m5}-{f4}.fits"


def download_one_fits(row, dr=FITS_DR, timeout=60, n_retries=3):
    """
    Download a single spec-lite FITS file with retry logic.

    Returns
    -------
    (status, path_or_error)
      status: 'exists' | 'ok' | 'fail'
    """
    plate = int(row["plate"])
    mjd   = int(row["mjd"])
    fiber = int(row["fiberid"])

    dest = fits_local_path(plate, mjd, fiber)
    dest.parent.mkdir(parents=True, exist_ok=True)   # create {plate:04d}/ subdir

    # ── Resume: skip if file already exists and is non-empty ───────────
    if dest.exists() and dest.stat().st_size > 10_000:   # 10 KB minimum sanity check
        return ("exists", dest)

    url = fits_url(plate, mjd, fiber, dr=dr)

    for attempt in range(n_retries):
        try:
            resp = requests.get(url, timeout=timeout, stream=True)
            if resp.status_code == 200:
                with open(dest, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=65536):
                        f.write(chunk)
                if dest.stat().st_size > 10_000:
                    return ("ok", dest)
                else:
                    dest.unlink(missing_ok=True)   # truncated file — retry
            elif resp.status_code == 404:
                return ("fail", f"404 Not Found: {url}")
            else:
                time.sleep(2 ** attempt)   # exponential back-off
        except Exception as e:
            time.sleep(2 ** attempt)

    return ("fail", f"All {n_retries} attempts failed: {url}")


# ── Count what we still need to download ───────────────────────────────
already_done = 0
to_download  = []
for _, row in df_sdss.iterrows():
    dest = fits_local_path(int(row["plate"]), int(row["mjd"]), int(row["fiberid"]))
    if dest.exists() and dest.stat().st_size > 10_000:
        already_done += 1
    else:
        to_download.append(row)

print(f"Already downloaded: {already_done:,} / {len(df_sdss):,}")
print(f"Still to download:  {len(to_download):,}")
if len(to_download) == 0:
    print("✓ All FITS files already present — nothing to do.")


In [ ]:
# ── Run the parallel download ─────────────────────────────────────────
# Only execute if there's work to do.
if len(to_download) == 0:
    print("✓ Nothing to download.")
else:
    log.info(f"Starting parallel download: {len(to_download):,} files, {N_WORKERS} workers")
    log.info(f"Estimated size: {len(to_download) * 0.2:.0f} MB "
             f"(assuming ~200 KB/file, varies widely)")

    results_ok   = []
    results_fail = []
    n_exists     = 0
    t0 = time.time()

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(download_one_fits, row): i
                   for i, row in enumerate(to_download)}

        for done_n, future in enumerate(as_completed(futures), start=1):
            status, info = future.result()
            if status == "exists":
                n_exists += 1
            elif status == "ok":
                results_ok.append(info)
            else:
                results_fail.append(info)

            # Progress report every 100 completions
            if done_n % 100 == 0 or done_n == len(to_download):
                elapsed = time.time() - t0
                rate    = done_n / elapsed
                remain  = (len(to_download) - done_n) / max(rate, 0.001)
                log.info(
                    f"  {done_n:>6,}/{len(to_download):,}  "
                    f"ok={len(results_ok):,}  fail={len(results_fail):,}  "
                    f"{rate:.1f}/s  ETA {remain/60:.1f} min"
                )

    total_time = time.time() - t0
    log.info(f"Download complete in {total_time/60:.1f} min")
    log.info(f"  OK:   {len(results_ok):,}")
    log.info(f"  Fail: {len(results_fail):,}")

    # Save a list of failed downloads for inspection / manual retry
    if results_fail:
        fail_log = SDSS_RAW_DIR / "failed_downloads.txt"
        with open(fail_log, "w") as f:
            f.write("\n".join(str(x) for x in results_fail))
        log.warning(f"{len(results_fail):,} failures logged to {fail_log}")
        log.warning("Re-run this cell to retry failures (they are skipped if already present).")


## Cell 6 — Backup: Generate wget Command File

If the Python parallel downloader is too slow or unreliable, this cell
generates a file of wget commands that can be run directly in bash,
taking advantage of shell-level parallelism and wget's own retry logic.

On your 768-core server:
```bash
# Run 64 parallel wget jobs (adjust -P to taste):
parallel -j 64 < sdss_raw/wget_commands.txt
```
Or with GNU xargs:
```bash
xargs -P 64 -a sdss_raw/wget_commands.txt sh -c
```


In [ ]:
# Generate wget commands only for files not yet present
wget_cmds = []
for _, row in df_sdss.iterrows():
    dest = fits_local_path(int(row["plate"]), int(row["mjd"]), int(row["fiberid"]))
    if not (dest.exists() and dest.stat().st_size > 10_000):
        url = fits_url(int(row["plate"]), int(row["mjd"]), int(row["fiberid"]))
        dest.parent.mkdir(parents=True, exist_ok=True)
        # -q: quiet  -c: resume partial  -P: output directory  --retry-connrefused
        wget_cmds.append(f"wget -q -c --tries=5 --retry-connrefused -O {dest} {url}")

wget_file = SDSS_RAW_DIR / "wget_commands.txt"
with open(wget_file, "w") as f:
    f.write("\n".join(wget_cmds))

print(f"Wrote {len(wget_cmds):,} wget commands to {wget_file}")
print(f"\nTo download with 64 parallel workers:")
print(f"  parallel -j 64 < {wget_file}")
print(f"\nTo download with xargs (no GNU parallel required):")
print(f"  xargs -P 64 -I CMD bash -c CMD < {wget_file}")
print(f"\nEstimated time: {len(wget_cmds)*0.3/64/60:.0f} min with 64 workers "
      f"(assumes 300ms avg latency)")


## Cell 7 — Download Progress Check

In [ ]:
# Verify what's on disk and check for suspiciously small files
n_ok = n_missing = n_small = 0
small_files = []

for _, row in df_sdss.iterrows():
    dest = fits_local_path(int(row["plate"]), int(row["mjd"]), int(row["fiberid"]))
    if not dest.exists():
        n_missing += 1
    elif dest.stat().st_size < 10_000:
        n_small += 1
        small_files.append(str(dest))
    else:
        n_ok += 1

total = len(df_sdss)
print(f"{'Status':<20s} {'Count':>8s}  {'Pct':>6s}")
print("-" * 38)
print(f"{'✓ OK (>10 KB)':<20s} {n_ok:>8,}  {100*n_ok/total:>5.1f}%")
print(f"{'✗ Missing':<20s} {n_missing:>8,}  {100*n_missing/total:>5.1f}%")
print(f"{'⚠ Too small (<10 KB)':<20s} {n_small:>8,}  {100*n_small/total:>5.1f}%")
print(f"{'Total in metadata':<20s} {total:>8,}  {'100.0':>5}%")

if n_small > 0:
    print(f"\nFirst 5 suspiciously small files:")
    for p in small_files[:5]:
        sz = Path(p).stat().st_size
        print(f"  {sz:>8,} bytes  {p}")
    print(f"  → These may be 404 error pages. Delete them and re-run Cell 5.")

if n_missing > 0:
    print(f"\n{n_missing:,} files still missing — re-run Cell 5 to download them.")
elif n_small == 0:
    print(f"\n✓ All {n_ok:,} FITS files present and non-trivial!")


## Cell 8 — Download WISE Cross-Match Photometry (SQL → CSV)

WISE photometry for SDSS galaxies is stored in two tables on the SDSS SkyServer:
- `wise_allsky` — the AllWISE source catalogue with W1/W2/W3/W4 magnitudes
- `wise_xmatch` — the astrometric cross-match table (SDSS ↔ WISE, max 2 arcsec)

The same paginated HTTP SQL approach works here. The join chain:
```
SpecObj → PhotoObj → wise_xmatch → wise_allsky
```

The WISE data is much smaller than the FITS spectra (~80 MB for 510k rows as CSV)
and downloads in a few minutes.


In [ ]:
def build_wise_sql_paged(z_max=0.5):
    """
    SQL template for paginated WISE cross-match retrieval.

    The {cursor} placeholder becomes `s.specobjid > {last_id}`.
    ORDER BY s.specobjid enables cursor-based pagination.
    """
    return f"""SELECT TOP {{page_size}}
    s.specobjid, s.plate, s.mjd, s.fiberid,
    s.z AS redshift, s.class AS specclass, s.subclass AS specsubclass,
    p.petroMag_r,
    w.w1mpro, w.w2mpro, w.w3mpro, w.w4mpro,
    w.w1sigmpro, w.w2sigmpro, w.w3sigmpro, w.w4sigmpro,
    w.w1snr,   w.w2snr,   w.w3snr,   w.w4snr
FROM SpecObj AS s
JOIN PhotoObj AS p ON s.bestobjid = p.objid
JOIN wise_xmatch AS x ON x.sdss_objid = p.objid
JOIN wise_allsky AS w ON x.wise_cntr  = w.cntr
WHERE {{cursor}}
  AND s.class = 'GALAXY'
  AND s.z BETWEEN 0.01 AND {z_max}
  AND s.zwarning = 0
  AND p.petroMag_r BETWEEN 14.0 AND 17.8
  AND p.petroMag_r != -9999
ORDER BY s.specobjid"""


if WISE_CSV.exists():
    existing_wise = pd.read_csv(WISE_CSV)
    log.info(f"Found existing {WISE_CSV}: {len(existing_wise):,} rows — reusing.")
    log.info(f"  Delete {WISE_CSV} to re-download.")
    df_wise = existing_wise
else:
    log.info(f"Downloading WISE cross-match data (N_SPECTRA={N_SPECTRA}) ...")
    t0 = time.time()

    wise_sql = build_wise_sql_paged(z_max=Z_MAX)
    df_wise = sql_query_paginated(
        wise_sql,
        id_col="specobjid",
        cursor_col="s.specobjid",   # qualified name avoids ambiguity in the JOIN
        page_size=PAGE_SIZE,
        max_rows=N_SPECTRA,
        dr=SQL_DR,
    )

    if len(df_wise) == 0:
        raise RuntimeError(
            "WISE query returned 0 rows.\n"
            "  Possible causes:\n"
            "  1. wise_xmatch table not in DR{SQL_DR} — try SQL_DR=16 or 17\n"
            "  2. Network issue — check connectivity first (Cell 2 test)\n"
            "  3. Query syntax error — run the test in Cell 2 first"
        )

    df_wise.to_csv(WISE_CSV, index=False)
    log.info(f"Saved {len(df_wise):,} rows to {WISE_CSV} in {time.time()-t0:.1f}s")

print(f"\nWISE data: {len(df_wise):,} galaxies × {len(df_wise.columns)} columns")
print(df_wise.head(3).to_string())

## Cell 9 — WISE Download Diagnostic

In [ ]:
print(f"WISE cross-match: {len(df_wise):,} galaxies")
print(f"Saved to: {WISE_CSV}")
print(f"File size: {WISE_CSV.stat().st_size/1e6:.1f} MB")
print(f"\nColumn types:")
print(df_wise.dtypes.to_string())

# Check expected WISE magnitude columns
w_cols = ["w1mpro","w2mpro","w3mpro","w4mpro"]
print(f"\nWISE magnitude ranges (non-null):")
for col in w_cols:
    if col in df_wise.columns:
        vals = pd.to_numeric(df_wise[col], errors="coerce").dropna()
        n_det = len(vals)
        pct = 100 * n_det / len(df_wise)
        print(f"  {col}: N_det={n_det:,} ({pct:.1f}%)  "
              f"range=[{vals.min():.2f}, {vals.max():.2f}]")


## Cell 10 — Summary & Next Steps

In [ ]:
# Count downloaded FITS files
n_fits_ok = sum(
    1 for _, row in df_sdss.iterrows()
    if fits_local_path(int(row["plate"]), int(row["mjd"]), int(row["fiberid"])).exists()
    and fits_local_path(int(row["plate"]), int(row["mjd"]), int(row["fiberid"])).stat().st_size > 10_000
)

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║              DOWNLOAD SUMMARY                                    ║
╠══════════════════════════════════════════════════════════════════╣
║  SDSS metadata CSV:   {str(SDSS_META_CSV):<40s} ║
║  SDSS galaxies:       {len(df_sdss):>8,}                               ║
║  FITS files on disk:  {n_fits_ok:>8,} / {len(df_sdss):,}                    ║
║                                                                  ║
║  WISE CSV:            {str(WISE_CSV):<40s} ║
║  WISE galaxies:       {len(df_wise):>8,}                               ║
╠══════════════════════════════════════════════════════════════════╣
║  NEXT STEPS                                                      ║
║                                                                  ║
║  1. Open sdss_data_preprocessing.ipynb                           ║
║     Set DATA_SOURCE = "local_fits"                               ║
║     Set FITS_DIR    = "sdss_raw/fits"                            ║
║     Run all cells → sdss_output/                                 ║
║                                                                  ║
║  2. Open wise_data_preprocessing.ipynb                           ║
║     Set DATA_SOURCE = "csv"                                      ║
║     Set CSV_PATH    = "wise_raw/wise_crossmatch.csv"             ║
║     Run all cells → wise_output/                                 ║
║                                                                  ║
║  3. Open reproduce_figure3_ppplot.ipynb                          ║
║     Run all cells (reads from sdss_output/ + wise_output/)       ║
╚══════════════════════════════════════════════════════════════════╝
""")
